In [0]:
spark.sql("DROP TABLE IF EXISTS de_mini_project.silver.product")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS de_mini_project.silver.product (
    product_id STRING,
    item_name STRING,
    category STRING,
    base_price DECIMAL(10,2),
    marked_price DECIMAL(10,2)
)
""")

In [0]:
spark.sql("""
INSERT INTO de_mini_project.silver.product (product_id, item_name, category, base_price, marked_price)
WITH product_cleaned AS (
    SELECT 
        product_id AS product_id,
        item_name_description AS item_name,
        INITCAP(_category_type_) AS category,
        CAST(base_cost AS DECIMAL(10,2)) AS base_price,
        CAST(_marked_price_ AS DECIMAL(10,2)) AS marked_price
    FROM de_mini_project.azure_blob_storage.products
)
SELECT 
    product_id, 
    item_name, 
    category, 
    base_price, 
    marked_price
FROM product_cleaned
QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY marked_price DESC) = 1
""")